# Notebook 03: Sampling Strategies

## Why This Matters
The transformer outputs a probability distribution over the entire vocabulary at each step. How you sample from that distribution determines whether your model writes coherent prose, hallucinates freely, or produces robotic repetition. Sampling parameters are the most frequently misunderstood LLM knobs — most practitioners cargo-cult values without knowing what they're actually doing.

This notebook builds every sampling method from scratch and gives you the intuition to reason about the tradeoffs for any use case: creative writing, code generation, factual Q&A, structured output.

## Structure
1. The logit distribution — what the model actually outputs
2. Greedy decoding — argmax, always
3. Temperature scaling — sharpen or flatten the distribution
4. Top-k sampling — restrict to the k most likely tokens
5. Top-p (nucleus) sampling — restrict to tokens covering p probability mass
6. Min-p sampling — a newer alternative to top-p
7. Combining strategies — temperature + top-p in practice
8. Repetition penalty
9. When to use what — practical guide

## What You'll Learn
- Why temperature=0 is greedy (not deterministic in the probabilistic sense)
- Why top-k has a fixed-k problem that nucleus sampling solves
- The entropy of a distribution and why it predicts generation quality
- How to visualize what each sampling parameter actually does to the distribution

## Background

### What is sampling?

At every step of generation, a transformer outputs a probability distribution over the entire vocabulary — tens of thousands of possible next tokens, each with a score. *Sampling* is the process of converting that distribution into a single token choice.

This sounds trivial — just pick the most likely token, right? In practice, how you sample determines whether your model produces coherent prose, robotic repetition, creative surprises, or complete nonsense. Sampling parameters are the most frequently misunderstood controls in LLM systems, and most practitioners cargo-cult values without understanding what they actually do.

### Why not always pick the most likely token?

The obvious choice — always pick the highest-probability token — is called **greedy decoding**. It's fast, deterministic, and often useful. But it has a critical failure mode: repetition.

If "the" is the most likely next token, greedy decoding picks it. Now in this new context, "the" is again likely — and we're in a loop. Real models trained on human text exhibit this pathology badly: given free rein, greedy-decoded LLMs often fall into phrase loops within a few dozen tokens.

More subtly, Holtzman et al. (2020) showed in *"The Curious Case of Neural Text Degeneration"* that **human text is not the most probable text**. When humans write, they regularly choose words that are informative but not maximally predictable. Maximum-probability text is bland, over-generic, and avoids specifics. Real writing lives in a moderate-probability region — specific enough to be interesting, not so rare as to be confusing.

### Temperature: the first control

Temperature comes from statistical mechanics (Boltzmann distributions). Dividing logits by a temperature `T` before softmax reshapes the distribution: `T < 1` sharpens it (tokens with high probability get even more weight), `T > 1` flattens it (probability spreads more uniformly). At `T → 0`, it converges to greedy (argmax). At `T → ∞`, it converges to uniform random choice.

Temperature was the standard approach for years. Its limitation: it uniformly adjusts all probabilities. At high temperature, you get more diversity, but also a non-trivial probability of sampling garbage tokens from the far tail of the distribution.

### Top-k and top-p: restricting the candidate pool

**Top-k sampling** (Fan et al., 2018) hardcodes a candidate pool size: only the `k` most probable tokens are eligible, all others are zeroed out. This prevents garbage tokens from the tail. But `k` is fixed regardless of how confident the model is — at a highly-predictable position, top-50 might include many very-low-probability junk tokens; at an ambiguous position, it might exclude many plausible options.

**Nucleus sampling / top-p** (Holtzman et al., 2020) solves this with an adaptive pool: keep the smallest set of tokens whose cumulative probability reaches `p`. When the model is confident, the nucleus is small (1–3 tokens); when uncertain, the nucleus expands. The pool adapts to the distribution's shape. This became the community standard almost immediately after publication, and remains so today.

### Why this matters for inference

Sampling happens at every single decode step — for a 500-token response, you make 500 sampling decisions. The parameters you choose (temperature, top-p) compound across the entire generation. Understanding them deeply is prerequisite to reasoning about output quality, reproducibility, and why the same prompt can produce radically different results on repeated calls.

In [ ]:
%pip install -q torch matplotlib numpy

In [ ]:
import math
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

torch.manual_seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

print("Imports OK")

## 1. The Logit Distribution

At each generation step, the model outputs a vector of **logits** — one raw score per vocabulary token. These are unnormalized log-probabilities.

```
logits: [-2.1,  0.3,  5.7,  1.2,  -0.8,  ...]   shape: (vocab_size,)
  ↓  softmax
probs:  [0.001, 0.01, 0.85, 0.05, 0.002, ...]   sums to 1.0
```

All sampling strategies are transformations applied to this distribution before drawing a token.

In [ ]:
# Simulate a realistic logit distribution:
# A few high-probability tokens, a long tail of low-probability ones
vocab_size = 50_257

# Create a peaked distribution: top few tokens dominate
logits = torch.randn(vocab_size) * 2.0
# Boost the top 5 tokens significantly
top_indices = torch.tensor([1024, 531, 8921, 42, 7777])
logits[top_indices] += torch.tensor([8.0, 6.0, 5.5, 4.5, 4.0])

probs = F.softmax(logits, dim=-1)

# Sort by probability
sorted_probs, sorted_indices = torch.sort(probs, descending=True)

print("Top 10 tokens:")
print(f"{'Rank':>5} {'Token ID':>10} {'Prob':>10} {'Cum Prob':>10}")
print("-" * 40)
cum = 0.0
for i in range(10):
    cum += sorted_probs[i].item()
    print(f"{i+1:>5} {sorted_indices[i].item():>10} {sorted_probs[i].item():>10.4f} {cum:>10.4f}")

print(f"\nTop 1 token captures {sorted_probs[0].item()*100:.1f}% of probability")
print(f"Top 10 tokens capture {sorted_probs[:10].sum().item()*100:.1f}% of probability")
print(f"Top 100 tokens capture {sorted_probs[:100].sum().item()*100:.1f}% of probability")

# Entropy of the distribution
entropy = -(probs * torch.log2(probs + 1e-10)).sum().item()
print(f"\nEntropy: {entropy:.2f} bits  (max possible: {math.log2(vocab_size):.1f} bits)")

## 2. Greedy Decoding

Always pick the highest-probability token. Deterministic, fast, and often repetitive.

```python
next_token = torch.argmax(logits)
```

**Problem:** Greedy decoding gets stuck. Once it generates a repetitive pattern, that pattern tends to have high probability, so it keeps repeating. Also misses the globally optimal sequence — the highest-probability next token doesn't always lead to the highest-probability overall sequence.

In [ ]:
def greedy_sample(logits: torch.Tensor) -> int:
    return torch.argmax(logits).item()


# Demonstrate the repetition problem with a toy transition matrix
# Simulates a model that gets stuck in a loop
print("Greedy decoding repetition example:")
print("Simulating: logits favor token A initially, then A→A transition becomes dominant")

# Toy: tokens are just integers 0-4, simulated transitions
transitions = {
    # From token: {to_token: logit}
    0: torch.tensor([1.0, 5.0, 2.0, 1.0, 1.0]),  # from 0, prefer 1
    1: torch.tensor([1.0, 1.0, 6.0, 1.0, 1.0]),  # from 1, prefer 2
    2: torch.tensor([1.0, 1.0, 1.0, 7.0, 1.0]),  # from 2, prefer 3
    3: torch.tensor([1.0, 1.0, 1.0, 1.0, 5.0]),  # from 3, prefer 4
    4: torch.tensor([1.0, 1.0, 1.0, 8.0, 1.0]),  # from 4, loops back to 3!
}

current = 0
sequence = [current]
for _ in range(15):
    current = greedy_sample(transitions[current])
    sequence.append(current)

print(f"Generated sequence: {sequence}")
print("Notice: enters 3→4→3→4... loop after step 3")

## 3. Temperature Scaling

Temperature `T` divides the logits before softmax:

$$p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

```
T → 0:    distribution collapses to argmax (greedy)
T = 1.0:  unmodified distribution
T > 1.0:  distribution flattens (more random, more diverse)
T → ∞:    uniform distribution (completely random)
```

Temperature doesn't change which tokens are possible — it changes how peaked the distribution is.

In [ ]:
def apply_temperature(logits: torch.Tensor, temperature: float) -> torch.Tensor:
    if temperature == 0:
        # Greedy: return one-hot at argmax
        one_hot = torch.zeros_like(logits)
        one_hot[torch.argmax(logits)] = 1.0
        return one_hot
    return F.softmax(logits / temperature, dim=-1)


temperatures = [0.01, 0.5, 1.0, 1.5, 2.0]

fig, axes = plt.subplots(1, len(temperatures), figsize=(15, 3), sharey=False)
fig.suptitle("Effect of Temperature on the Top-50 Token Distribution", fontsize=12)

for ax, T in zip(axes, temperatures):
    probs_T = apply_temperature(logits, T)
    top50 = probs_T.topk(50).values.numpy()
    entropy_T = -(probs_T * torch.log2(probs_T + 1e-10)).sum().item()

    ax.bar(range(50), top50, color='steelblue', alpha=0.8)
    ax.set_title(f"T={T}\nentropy={entropy_T:.1f} bits")
    ax.set_xlabel("Rank")
    if T == temperatures[0]:
        ax.set_ylabel("Probability")
    ax.tick_params(labelbottom=False)

plt.tight_layout()
plt.savefig("temperature_effect.png", bbox_inches='tight')
plt.show()
print("\nLow T: mass concentrated on top token (peaked)")
print("High T: mass spread across many tokens (flat)")

In [ ]:
# Temperature sampling function
def temperature_sample(logits: torch.Tensor, temperature: float = 1.0) -> int:
    if temperature == 0:
        return torch.argmax(logits).item()
    scaled_logits = logits / temperature
    probs = F.softmax(scaled_logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).item()


# Show distribution of sampled tokens over 1000 draws
n_samples = 1000
print(f"Top-3 token sampling frequency over {n_samples} draws:")
print(f"{'Temperature':>12} | ", end="")
top3 = sorted_indices[:3].tolist()
for t in top3:
    print(f"Token {t:>5} | ", end="")
print("Other")
print("-" * 70)

for T in [0.5, 1.0, 1.5, 2.0]:
    counts = {t: 0 for t in top3}
    other = 0
    for _ in range(n_samples):
        tok = temperature_sample(logits, T)
        if tok in counts:
            counts[tok] += 1
        else:
            other += 1
    print(f"{T:>12} | ", end="")
    for t in top3:
        print(f"{counts[t]/n_samples:>13.1%} | ", end="")
    print(f"{other/n_samples:.1%}")

## 4. Top-k Sampling

Keep only the `k` highest-probability tokens, zero out the rest, then sample.

```
k=1:  always picks the top token (effectively greedy)
k=10: samples uniformly-ish from the top 10 tokens
k=50: the standard recommendation for many tasks
```

**Problem:** `k` is a fixed count. When the model is confident (peaked distribution), top-50 might include many very-low-probability junk tokens. When the model is uncertain (flat distribution), top-50 might exclude many reasonable tokens. The right `k` varies by context — nucleus sampling solves this.

In [ ]:
def top_k_sample(logits: torch.Tensor, k: int, temperature: float = 1.0) -> int:
    # Zero out all tokens below the top-k threshold
    if k < logits.size(-1):
        kth_value = torch.topk(logits, k).values[-1]  # value of the k-th token
        logits = logits.clone()
        logits[logits < kth_value] = float('-inf')

    scaled = logits / max(temperature, 1e-8)
    probs = F.softmax(scaled, dim=-1)
    return torch.multinomial(probs, num_samples=1).item()


# Demonstrate the fixed-k problem
# Case 1: Peaked distribution — many low-prob tokens in k=50
peaked_logits = torch.randn(vocab_size) - 5
peaked_logits[0] = 10.0  # overwhelmingly likely token
peaked_probs = F.softmax(peaked_logits, dim=-1)

# Case 2: Flat distribution — top-50 misses many good tokens
flat_logits = torch.randn(vocab_size) * 0.1  # nearly uniform
flat_probs = F.softmax(flat_logits, dim=-1)

for name, p in [("Peaked dist", peaked_probs), ("Flat dist", flat_probs)]:
    sorted_p = p.sort(descending=True).values
    cum50 = sorted_p[:50].sum().item()
    entropy = -(p * torch.log2(p + 1e-10)).sum().item()
    print(f"{name}:")
    print(f"  Entropy:              {entropy:.2f} bits")
    print(f"  Top-1 probability:    {sorted_p[0].item():.4f}")
    print(f"  Top-50 cum. prob:     {cum50:.4f}")
    print(f"  Prob in rank 51-100:  {sorted_p[50:100].sum().item():.6f}")
    print()

## 5. Top-p (Nucleus) Sampling

Instead of a fixed count, keep the **smallest set of tokens whose cumulative probability ≥ p**. This set is the "nucleus."

```
Peaked distribution (model is confident):
  p=0.9 → nucleus might contain only 2-3 tokens

Flat distribution (model is uncertain):
  p=0.9 → nucleus might contain 500+ tokens
```

The nucleus adapts to the distribution's shape — larger when the model is uncertain, smaller when it's confident. Holtzman et al. (2020) showed this produces more human-like text than top-k.

Typical values: `p=0.9` to `p=0.95` for most tasks.

In [ ]:
def top_p_sample(logits: torch.Tensor, p: float, temperature: float = 1.0) -> int:
    scaled = logits / max(temperature, 1e-8)
    probs = F.softmax(scaled, dim=-1)

    # Sort probabilities descending
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # Remove tokens once cumulative probability exceeds p
    # Shift by 1 to include the token that pushes cumsum over p
    remove = cumulative_probs - sorted_probs > p
    sorted_probs[remove] = 0.0

    # Renormalize
    sorted_probs /= sorted_probs.sum()

    # Sample from the filtered distribution
    sampled_idx = torch.multinomial(sorted_probs, num_samples=1)
    return sorted_indices[sampled_idx].item()


def nucleus_size(logits: torch.Tensor, p: float, temperature: float = 1.0) -> int:
    """Return the number of tokens in the nucleus for a given p."""
    probs = F.softmax(logits / max(temperature, 1e-8), dim=-1)
    sorted_probs = torch.sort(probs, descending=True).values
    cumsum = torch.cumsum(sorted_probs, dim=-1)
    return (cumsum <= p).sum().item() + 1


print("Nucleus size adapts to distribution shape:")
print(f"{'Distribution':>15} {'Entropy':>8} | ", end="")
for p in [0.8, 0.9, 0.95, 0.99]:
    print(f"p={p:>4} | ", end="")
print()
print("-" * 75)

test_cases = [
    ("Very peaked",  peaked_logits),
    ("Our example",  logits),
    ("Flat",         flat_logits),
]

for name, lgt in test_cases:
    probs = F.softmax(lgt, dim=-1)
    entropy = -(probs * torch.log2(probs + 1e-10)).sum().item()
    print(f"{name:>15} {entropy:>8.2f} | ", end="")
    for p in [0.8, 0.9, 0.95, 0.99]:
        n = nucleus_size(lgt, p)
        print(f"{n:>6} | ", end="")
    print()

In [ ]:
# Visualize nucleus vs top-k on the same distribution
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Top-k vs Top-p: Tokens Kept (shown in blue)")

probs_sorted = sorted_probs[:100].numpy()  # top 100 for visibility
ranks = np.arange(1, 101)

# Top-50
axes[0].bar(ranks, probs_sorted, color=['steelblue' if i < 50 else 'lightgray' for i in range(100)])
axes[0].set_title(f"Top-k (k=50)\n{50} tokens kept")
axes[0].set_xlabel("Rank")
axes[0].set_ylabel("Probability")

# Top-p=0.9
n90 = nucleus_size(logits, 0.9)
axes[1].bar(ranks, probs_sorted, color=['steelblue' if i < n90 else 'lightgray' for i in range(100)])
axes[1].set_title(f"Top-p (p=0.9)\n{n90} tokens kept")
axes[1].set_xlabel("Rank")

# Top-p=0.95
n95 = nucleus_size(logits, 0.95)
axes[2].bar(ranks, probs_sorted, color=['steelblue' if i < n95 else 'lightgray' for i in range(100)])
axes[2].set_title(f"Top-p (p=0.95)\n{n95} tokens kept")
axes[2].set_xlabel("Rank")

plt.tight_layout()
plt.savefig("topk_vs_topp.png", bbox_inches='tight')
plt.show()

## 6. Min-p Sampling

Min-p (Nguyen et al., 2024) is a newer alternative to top-p. Instead of a cumulative probability threshold, it uses a **minimum probability threshold** relative to the top token:

```
min_p_threshold = min_p × p_top
Keep tokens where: p_token ≥ min_p_threshold
```

This scales the cutoff with the model's confidence. If the top token has 90% probability, the bar to be included is high. If the top token has 5% probability, many tokens can qualify.

Advantage over top-p: avoids cutting off too many tokens when the distribution is very peaked (top-p with p=0.9 might only keep 1-2 tokens), or keeping too many when it's flat.

In [ ]:
def min_p_sample(logits: torch.Tensor, min_p: float, temperature: float = 1.0) -> int:
    probs = F.softmax(logits / max(temperature, 1e-8), dim=-1)
    p_top = probs.max().item()
    threshold = min_p * p_top

    # Keep only tokens above threshold
    filtered = probs.clone()
    filtered[filtered < threshold] = 0.0
    filtered /= filtered.sum()  # renormalize

    return torch.multinomial(filtered, num_samples=1).item()


def min_p_nucleus_size(logits: torch.Tensor, min_p: float, temperature: float = 1.0) -> int:
    probs = F.softmax(logits / max(temperature, 1e-8), dim=-1)
    threshold = min_p * probs.max().item()
    return (probs >= threshold).sum().item()


print("Min-p nucleus size vs top-p nucleus size:")
print(f"{'Distribution':>15} | top-p 0.9 | top-p 0.95 | min-p 0.05 | min-p 0.1")
print("-" * 65)
for name, lgt in test_cases:
    n_p90 = nucleus_size(lgt, 0.9)
    n_p95 = nucleus_size(lgt, 0.95)
    n_mp05 = min_p_nucleus_size(lgt, 0.05)
    n_mp10 = min_p_nucleus_size(lgt, 0.10)
    print(f"{name:>15} | {n_p90:>9} | {n_p95:>10} | {n_mp05:>10} | {n_mp10:>9}")

## 7. Repetition Penalty

To reduce repetitive output, apply a penalty to tokens that have already appeared in the context.

```
If token t has appeared in context:
    logit[t] = logit[t] / penalty   if logit[t] > 0
    logit[t] = logit[t] * penalty   if logit[t] < 0
```

Typical values: `1.1` to `1.3`. At `1.0` there's no effect. Too high and the model avoids repeating necessary words ("the", "a", etc.).

In [ ]:
def apply_repetition_penalty(
    logits: torch.Tensor,
    input_ids: list[int],
    penalty: float = 1.0
) -> torch.Tensor:
    if penalty == 1.0 or not input_ids:
        return logits

    logits = logits.clone()
    for token_id in set(input_ids):
        if logits[token_id] > 0:
            logits[token_id] /= penalty
        else:
            logits[token_id] *= penalty
    return logits


# Show effect on token probabilities
context_tokens = sorted_indices[:5].tolist()  # top-5 most likely tokens have been generated
print(f"Context tokens (previously generated): {context_tokens}")
print(f"\n{'Token':>8} {'Base prob':>12} {'rep=1.1':>10} {'rep=1.3':>10} {'rep=1.5':>10}")
print("-" * 55)

for rank, tok in enumerate(sorted_indices[:8].tolist()):
    base_prob = F.softmax(logits, dim=-1)[tok].item()
    probs_by_pen = []
    for pen in [1.1, 1.3, 1.5]:
        penalized = apply_repetition_penalty(logits, context_tokens, pen)
        p = F.softmax(penalized, dim=-1)[tok].item()
        probs_by_pen.append(p)
    in_ctx = "*" if tok in context_tokens else " "
    print(f"{tok:>7}{in_ctx} {base_prob:>12.4f} {probs_by_pen[0]:>10.4f} {probs_by_pen[1]:>10.4f} {probs_by_pen[2]:>10.4f}")

print("\n* = token appeared in context (penalized)")

## 8. Putting It Together: A Complete Sampling Function

Production samplers apply these operations in order:
1. Repetition penalty
2. Temperature
3. Top-k filter
4. Top-p filter
5. Sample

In [ ]:
def sample_next_token(
    logits: torch.Tensor,
    temperature: float = 1.0,
    top_k: int = 0,
    top_p: float = 1.0,
    repetition_penalty: float = 1.0,
    input_ids: list[int] = None,
) -> int:
    """
    Full sampling pipeline used in production inference engines.
    Operations applied in order: rep penalty → temperature → top-k → top-p → sample.
    """
    # 1. Repetition penalty
    if repetition_penalty != 1.0 and input_ids:
        logits = apply_repetition_penalty(logits, input_ids, repetition_penalty)

    # 2. Temperature
    if temperature == 0:
        return torch.argmax(logits).item()
    logits = logits / temperature

    # 3. Top-k filter
    if top_k > 0 and top_k < logits.size(-1):
        kth = torch.topk(logits, top_k).values[-1]
        logits = logits.clone()
        logits[logits < kth] = float('-inf')

    # 4. Top-p filter
    if top_p < 1.0:
        probs_sorted, indices_sorted = torch.sort(F.softmax(logits, dim=-1), descending=True)
        cumsum = torch.cumsum(probs_sorted, dim=-1)
        remove = cumsum - probs_sorted > top_p
        remove_mask = torch.zeros_like(logits, dtype=torch.bool)
        remove_mask[indices_sorted[remove]] = True
        logits = logits.clone()
        logits[remove_mask] = float('-inf')

    # 5. Sample
    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).item()


# Test with common presets
presets = [
    ("Greedy",         dict(temperature=0)),
    ("Creative",       dict(temperature=0.9, top_p=0.95)),
    ("Balanced",       dict(temperature=0.7, top_p=0.9)),
    ("Conservative",   dict(temperature=0.3, top_k=20)),
    ("Code gen",       dict(temperature=0.2, top_p=0.95, repetition_penalty=1.1)),
]

print("Sampling 100 tokens from each preset — top-3 most frequent:")
print(f"{'Preset':>15} | Top token | 2nd token | 3rd token | Diversity")
print("-" * 65)
for name, kwargs in presets:
    counts = {}
    for _ in range(200):
        tok = sample_next_token(logits.clone(), **kwargs)
        counts[tok] = counts.get(tok, 0) + 1
    top3 = sorted(counts.items(), key=lambda x: -x[1])[:3]
    diversity = len(counts)
    top3_str = "  ".join(f"{t}:{c/200:.0%}" for t, c in top3)
    print(f"{name:>15} | {top3_str:<40} | {diversity} unique")

## 9. When to Use What

| Use case | Temperature | Top-p | Top-k | Notes |
|----------|-------------|-------|-------|-------|
| Factual Q&A | 0–0.3 | — | — | Low temperature or greedy; wrong facts are worse than boring prose |
| Code generation | 0.2–0.4 | 0.95 | — | Low temp + nucleus; repetition_penalty=1.1 helps |
| Creative writing | 0.7–1.0 | 0.9–0.95 | — | Standard nucleus sampling |
| Chat / assistant | 0.6–0.8 | 0.9 | — | Balanced; avoid extremes |
| Structured output (JSON) | 0.0–0.2 | — | — | Or use constrained decoding (notebook 15) |
| Poetry / brainstorm | 0.9–1.2 | 0.95 | — | High temperature explores distribution tail |
| Summarization | 0.3–0.5 | 0.9 | — | Some diversity but needs fidelity |

**Rules of thumb:**
- Temperature and top-p are the two main levers; top-k is largely superseded by top-p
- Never use temperature alone at high values without top-p — you'll sample garbage tokens
- `temperature=0` for anything requiring exact correctness
- `top_p=0.9, temperature=0.7` is a safe default for most text generation tasks
- Entropy of the distribution is the best diagnostic: low entropy = model is confident = low temp appropriate

In [ ]:
# Final: entropy as a diagnostic for sampling parameter selection
print("Entropy guide for sampling parameter selection:")
print()

scenarios = [
    ("Factual recall (confident model)", torch.randn(vocab_size) - 3.0),
    ("Typical text generation",          logits),
    ("Ambiguous/creative context",        torch.randn(vocab_size) * 0.5),
]

for name, lgt in scenarios:
    p = F.softmax(lgt, dim=-1)
    entropy = -(p * torch.log2(p + 1e-10)).sum().item()
    top1 = p.max().item()
    print(f"  {name}")
    print(f"    Entropy: {entropy:.1f} bits  |  Top-1 prob: {top1:.3f}")
    if entropy < 3:
        print(f"    → Model is confident. Use low temperature (0.1–0.4) or greedy.")
    elif entropy < 8:
        print(f"    → Moderate uncertainty. Use temperature 0.6–0.8 + top-p 0.9.")
    else:
        print(f"    → High uncertainty. Model is unsure; quality may be low regardless.")
    print()

## Summary

| Method | Mechanism | Key parameter | Best for |
|--------|-----------|---------------|----------|
| Greedy | argmax | — | Deterministic, factual tasks |
| Temperature | logit scaling | T ∈ (0, 2] | All tasks; adjust sharpness |
| Top-k | keep top k | k ∈ [1, vocab] | Simple truncation |
| Top-p (nucleus) | keep top cumulative-p | p ∈ (0, 1] | Adaptive; standard choice |
| Min-p | threshold relative to top token | min_p ∈ (0, 1] | Better for peaked distributions |
| Rep. penalty | penalize seen tokens | penalty > 1.0 | Reducing repetition |

**Next:** Notebook 04 covers decoding algorithms — beam search, contrastive search, and how they differ from token-level sampling.